# Stage 2

## Tentukan Tujuan Model dan Metrik Evaluasi
Tentukan tipe problem: klasifikasi <br>
Metrik : F1-Score, ROC-AUC karena data yang digunakan imbalanced.

In [308]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [309]:
df_general = pd.read_csv('dataset/general_data.csv')
df_employee = pd.read_csv('dataset/employee_survey_data.csv')
df_manager = pd.read_csv('dataset/manager_survey_data.csv')
df_in_time = pd.read_csv('dataset/in_time.csv')
df_out_time = pd.read_csv('dataset/out_time.csv')

In [310]:
cols_to_fix = df_in_time.columns[1:]

df_in_time[cols_to_fix] = df_in_time[cols_to_fix].apply(pd.to_datetime)
df_out_time[cols_to_fix] = df_out_time[cols_to_fix].apply(pd.to_datetime)
work_hours = df_out_time.iloc[:, 1:] - df_in_time.iloc[:, 1:]

In [311]:
work_hours = (df_out_time.iloc[:, 1:] - df_in_time.iloc[:, 1:]) / pd.Timedelta(hours=1)
total_work_hours = work_hours.sum(axis=1)
average_work_hours = work_hours.mean(axis=1)

df_in_time.iloc[:, 1:] = df_in_time.iloc[:, 1:].apply(pd.to_datetime, errors='coerce')
df_out_time.iloc[:, 1:] = df_out_time.iloc[:, 1:].apply(pd.to_datetime, errors='coerce')

in_long = df_in_time.melt(id_vars=['Unnamed: 0'],
                     var_name='date',
                     value_name='in_time')

out_long = df_out_time.melt(id_vars=['Unnamed: 0'],
                       var_name='date',
                       value_name='out_time')

df = in_long.merge(out_long, on=['Unnamed: 0', 'date'])

df = df.dropna(subset=['in_time', 'out_time'])

df['work_hours'] = (df['out_time'] - df['in_time']).dt.total_seconds() / 3600

df = df[df['work_hours'] >= 0]

df['is_overwork'] = df['work_hours'] >= 9

overwork_days = df.groupby('Unnamed: 0')['is_overwork'].sum()
overwork_days = overwork_days.reset_index(drop=True)

In [312]:
df_in_out = pd.concat(
    [total_work_hours, average_work_hours, overwork_days],
    axis=1
)

df_in_out.columns = ['total_work_hours', 'average_work_hours', 'overwork_days']
df_in_out = df_in_out.reset_index()

In [313]:
df_in_out['overwork'] = df_in_out['total_work_hours'].apply(
    lambda x: 'Yes' if x > 2080 else 'No'
)

In [314]:
df_in_out = df_in_out.drop(columns=['index'])

df_in_out.insert(0, 'EmployeeID', range(1, len(df_in_out) + 1))

In [315]:
final_df = pd.merge(df_general, df_employee, on='EmployeeID')
final_df = pd.merge(final_df,df_manager, on='EmployeeID')
final_df = pd.merge(final_df,df_in_out, on='EmployeeID')

In [316]:
final_df['Attrition'] = final_df['Attrition'].map({'Yes': 1, 'No': 0})
final_df['isMale'] = final_df['Gender'].map({'Male': 1, 'Female': 0})
final_df['overwork'] = final_df['overwork'].map({'Yes': 1, 'No': 0})

In [317]:
final_df = pd.get_dummies(final_df, columns=['Department'],dtype=int, drop_first=True)
final_df = pd.get_dummies(final_df, columns=['EducationField'],dtype=int, drop_first=True)
final_df = pd.get_dummies(final_df, columns=['JobRole'],dtype=int, drop_first=True)
final_df = pd.get_dummies(final_df, columns=['MaritalStatus'],dtype=int, drop_first=True)
final_df = pd.get_dummies(final_df, columns=['BusinessTravel'],dtype=int, drop_first=True)

## Split Dataset & Preprocessing

In [318]:
from sklearn.model_selection import train_test_split

y = final_df['Attrition']
X = final_df.drop(columns=['Attrition'])

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2,
    random_state= 42, 
    stratify=y
)

In [319]:
X_train.fillna({'EnvironmentSatisfaction':X_train['EnvironmentSatisfaction'].mode()[0]}, inplace=True)
X_train.fillna({'JobSatisfaction':X_train['JobSatisfaction'].mode()[0]}, inplace=True)
X_train.fillna({'WorkLifeBalance':X_train['WorkLifeBalance'].mode()[0]}, inplace=True)
X_train.fillna({'NumCompaniesWorked':X_train['NumCompaniesWorked'].mode()[0]}, inplace=True)
X_train.fillna({'TotalWorkingYears':X_train['TotalWorkingYears'].mode()[0]}, inplace=True)

X_val.fillna({'EnvironmentSatisfaction':X_val['EnvironmentSatisfaction'].mode()[0]}, inplace=True)
X_val.fillna({'JobSatisfaction':X_val['JobSatisfaction'].mode()[0]}, inplace=True)
X_val.fillna({'WorkLifeBalance':X_val['WorkLifeBalance'].mode()[0]}, inplace=True)
X_val.fillna({'NumCompaniesWorked':X_val['NumCompaniesWorked'].mode()[0]}, inplace=True)
X_val.fillna({'TotalWorkingYears':X_val['TotalWorkingYears'].mode()[0]}, inplace=True)

X.fillna({'EnvironmentSatisfaction':X['EnvironmentSatisfaction'].mode()[0]}, inplace=True)
X.fillna({'JobSatisfaction':X['JobSatisfaction'].mode()[0]}, inplace=True)
X.fillna({'WorkLifeBalance':X['WorkLifeBalance'].mode()[0]}, inplace=True)
X.fillna({'NumCompaniesWorked':X['NumCompaniesWorked'].mode()[0]}, inplace=True)
X.fillna({'TotalWorkingYears':X['TotalWorkingYears'].mode()[0]}, inplace=True)

,Age,DistanceFromHome,Education,EmployeeCount,EmployeeID,Gender,JobLevel,MonthlyIncome,NumCompaniesWorked,Over18,...,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely
0,51,6,2,1,1,Female,1,131160,1.0,Y,...,0,0,0,0,0,0,1,0,0,1
1,31,10,1,1,2,Female,1,41890,0.0,Y,...,0,0,0,1,0,0,0,1,1,0
2,32,17,4,1,3,Male,4,193280,1.0,Y,...,0,0,0,0,1,0,1,0,1,0
3,38,2,5,1,4,Male,3,83210,3.0,Y,...,0,0,0,0,0,0,1,0,0,0
4,32,10,1,1,5,Male,1,23420,4.0,Y,...,0,0,0,0,1,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4405,42,5,4,1,4406,Female,1,60290,3.0,Y,...,0,0,0,1,0,0,0,1,0,1
4406,29,2,4,1,4407,Male,1,26790,2.0,Y,...,0,0,0,0,0,0,0,0,0,1
4407,25,25,2,1,4408,Male,2,37020,0.0,Y,...,0,0,0,0,1,0,1,0,0,1
4408,42,18,2,1,4409,Male,1,23980,0.0,Y,...,0,0,0,0,0,0,0,0,0,1


In [320]:
X_train['DistanceFromHome_log'] = np.log1p(X_train['DistanceFromHome'])
X_train['MonthlyIncome_log'] = np.log1p(X_train['MonthlyIncome'])
X_train['NumCompaniesWorked_log'] = np.log1p(X_train['NumCompaniesWorked'])
X_train['PercentSalaryHike_log'] = np.log1p(X_train['PercentSalaryHike'])
X_train['TotalWorkingYears_log'] = np.log1p(X_train['TotalWorkingYears'])
X_train['YearsAtCompany_log'] = np.log1p(X_train['YearsAtCompany'])
X_train['YearsSinceLastPromotion_log'] = np.log1p(X_train['YearsSinceLastPromotion'])
X_train['YearsWithCurrManager_log'] = np.log1p(X_train['YearsWithCurrManager'])

X_val['DistanceFromHome_log'] = np.log1p(X_val['DistanceFromHome'])
X_val['MonthlyIncome_log'] = np.log1p(X_val['MonthlyIncome'])
X_val['NumCompaniesWorked_log'] = np.log1p(X_val['NumCompaniesWorked'])
X_val['PercentSalaryHike_log'] = np.log1p(X_val['PercentSalaryHike'])
X_val['TotalWorkingYears_log'] = np.log1p(X_val['TotalWorkingYears'])
X_val['YearsAtCompany_log'] = np.log1p(X_val['YearsAtCompany'])
X_val['YearsSinceLastPromotion_log'] = np.log1p(X_val['YearsSinceLastPromotion'])
X_val['YearsWithCurrManager_log'] = np.log1p(X_val['YearsWithCurrManager'])

X['DistanceFromHome_log'] = np.log1p(X['DistanceFromHome'])
X['MonthlyIncome_log'] = np.log1p(X['MonthlyIncome'])
X['NumCompaniesWorked_log'] = np.log1p(X['NumCompaniesWorked'])
X['PercentSalaryHike_log'] = np.log1p(X['PercentSalaryHike'])
X['TotalWorkingYears_log'] = np.log1p(X['TotalWorkingYears'])
X['YearsAtCompany_log'] = np.log1p(X['YearsAtCompany'])
X['YearsSinceLastPromotion_log'] = np.log1p(X['YearsSinceLastPromotion'])
X['YearsWithCurrManager_log'] = np.log1p(X['YearsWithCurrManager'])

In [321]:
X_train = X_train.drop(columns= ['EmployeeID','DistanceFromHome','Gender','MonthlyIncome','NumCompaniesWorked','PercentSalaryHike',
                                   'TotalWorkingYears','YearsAtCompany','YearsAtCompany','YearsSinceLastPromotion','YearsWithCurrManager',
                                   'TotalWorkingYears_log','YearsSinceLastPromotion_log','YearsWithCurrManager_log',
                                   'average_work_hours','overwork_days','overwork','StandardHours','EmployeeCount','Over18'])
X_val = X_val.drop(columns= ['EmployeeID','DistanceFromHome','Gender','MonthlyIncome','NumCompaniesWorked','PercentSalaryHike',
                                   'TotalWorkingYears','YearsAtCompany','YearsAtCompany','YearsSinceLastPromotion','YearsWithCurrManager',
                                   'TotalWorkingYears_log','YearsSinceLastPromotion_log','YearsWithCurrManager_log',
                                   'average_work_hours','overwork_days','overwork','StandardHours','EmployeeCount','Over18'])
X = X.drop(columns= ['EmployeeID','DistanceFromHome','Gender','MonthlyIncome','NumCompaniesWorked','PercentSalaryHike',
                                   'TotalWorkingYears','YearsAtCompany','YearsAtCompany','YearsSinceLastPromotion','YearsWithCurrManager',
                                   'TotalWorkingYears_log','YearsSinceLastPromotion_log','YearsWithCurrManager_log',
                                   'average_work_hours','overwork_days','overwork','StandardHours','EmployeeCount','Over18'])

## Modeling

In [322]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline 
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score

In [323]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import cross_validate

def eval_classification(model):
    # prediction
    y_pred = model.predict(X_val)
    y_pred_train = model.predict(X_train)

    # probability
    y_proba = model.predict_proba(X_val)[:,1]
    y_proba_train = model.predict_proba(X_train)[:,1]

    # metrics
    print("Accuracy (val Set): %.2f" % accuracy_score(y_val, y_pred))
    print("Accuracy (Train Set): %.2f" % accuracy_score(y_train, y_pred_train))
    print("Precision (val Set): %.2f" % precision_score(y_val, y_pred))
    print("Recall (val Set): %.2f" % recall_score(y_val, y_pred))
    print("F1-Score (val Set): %.2f" % f1_score(y_val, y_pred))
    print("roc_auc (val): %.2f" % roc_auc_score(y_val, y_proba))
    print("roc_auc (train): %.2f" % roc_auc_score(y_train, y_proba_train))

    score = cross_validate(model, X, y, cv=5, scoring='recall', return_train_score=True)
    print('recall (cv train):', score['train_score'].mean())
    print('recall (cv test):', score['test_score'].mean())

def show_feature_importance(model):
    feat_importances = pd.Series(model.feature_importances_, index=X_train.columns)
    ax = feat_importances.nlargest(25).plot(kind='barh', figsize=(10, 8))
    ax.invert_yaxis()

    plt.xlabel('score')
    plt.ylabel('feature')
    plt.title('feature importance score')

def show_best_hyperparameter(model):
    print(model.best_estimator_.get_params())

### Logistic Regression

In [342]:
pipeline_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

pipeline_lr.fit(X_train, y_train)

y_prob = pipeline_lr.predict_proba(X_val)[:, 1]
y_pred_custom = (y_prob >= 0.5).astype(int)
roc_auc = roc_auc_score(y_val, y_pred_custom)
y_pred = pipeline_lr.predict(X_val)
print(classification_report(y_val, y_pred_custom))

              precision    recall  f1-score   support

           0       0.87      0.97      0.92       740
           1       0.61      0.22      0.32       142

    accuracy                           0.85       882
   macro avg       0.74      0.60      0.62       882
weighted avg       0.82      0.85      0.82       882



In [346]:
eval_classification(pipeline_lr)

Accuracy (val Set): 0.85
Accuracy (Train Set): 0.86
Precision (val Set): 0.61
Recall (val Set): 0.22
F1-Score (val Set): 0.32
roc_auc (val): 0.77
roc_auc (train): 0.81
recall (cv train): 0.23910121537661824
recall (cv test): 0.2292524377031419


In [325]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_val, pipeline_lr.predict(X_val)))

[[720  20]
 [111  31]]


#### Parameter Tuning

In [326]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

logreg_tuned = LogisticRegression(random_state=42, max_iter=1000)

param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'l1_ratio': [0,1],
    'solver': ['liblinear']
}

grid_search_lr = GridSearchCV(
    estimator=logreg_tuned,
    param_grid=param_grid_lr,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

# training pakai SMOTE result
grid_search_lr.fit(X_train, y_train)

# prediction pakai test yang sudah di-scale
y_pred_lr = grid_search_lr.best_estimator_.predict(X_val)

print(classification_report(y_val, y_pred_lr))
print("Parameter Terbaik:",grid_search_lr.best_params_)

              precision    recall  f1-score   support

           0       0.87      0.97      0.92       740
           1       0.62      0.23      0.33       142

    accuracy                           0.85       882
   macro avg       0.74      0.60      0.62       882
weighted avg       0.83      0.85      0.82       882

Parameter Terbaik: {'C': 10, 'l1_ratio': 1, 'solver': 'liblinear'}


In [345]:
print(eval_classification(grid_search_lr))

Accuracy (val Set): 0.85
Accuracy (Train Set): 0.86
Precision (val Set): 0.62
Recall (val Set): 0.23
F1-Score (val Set): 0.33
roc_auc (val): 0.77
roc_auc (train): 0.81
recall (cv train): 0.2348808138815317
recall (cv test): 0.23067073771299124
None


In [327]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_val, grid_search_lr.predict(X_val)))

[[720  20]
 [110  32]]


### Decision Tree

In [350]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

pipeline_dt = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('model', DecisionTreeClassifier(max_depth=5, 
                                     min_samples_leaf=20, 
                                     random_state=42,
                                     ))
])
pipeline_dt.fit(X_train, y_train)
y_prob = pipeline_dt.predict_proba(X_val)[:, 1]
y_pred_custom = (y_prob >= 0.25).astype(int)
roc_auc = roc_auc_score(y_val, y_pred_custom)
y_pred = pipeline_dt.predict(X_val)
print(classification_report(y_val, y_pred_custom))
print("ROC AUC:", roc_auc)

              precision    recall  f1-score   support

           0       0.92      0.70      0.80       740
           1       0.30      0.68      0.42       142

    accuracy                           0.70       882
   macro avg       0.61      0.69      0.61       882
weighted avg       0.82      0.70      0.73       882

ROC AUC: 0.6887038446897602


In [351]:
eval_classification(pipeline_dt)

Accuracy (val Set): 0.81
Accuracy (Train Set): 0.83
Precision (val Set): 0.42
Recall (val Set): 0.52
F1-Score (val Set): 0.46
roc_auc (val): 0.75
roc_auc (train): 0.81
recall (cv train): 0.7292618629173989
recall (cv test): 0.6905840638235005


In [352]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_val, pipeline_dt.predict(X_val)))

[[637 103]
 [ 68  74]]


#### Parameter Tuning

In [330]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV

pipeline = Pipeline([
    ('model', DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42))
])

param_dist = {
    'model__max_depth': [3, 5, 7, 10],
    'model__min_samples_split': [5, 10, 20],
    'model__min_samples_leaf': [2, 5, 10],
    'model__criterion': ['gini', 'entropy']
}

random_search_dt = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='recall',
    random_state=42,
    n_jobs=-1
)

random_search_dt.fit(X_train, y_train)

y_pred = random_search_dt.predict(X_val)

print(classification_report(y_val, y_pred))
print("Parameter Terbaik:", random_search_dt.best_params_)

              precision    recall  f1-score   support

           0       0.94      0.96      0.95       740
           1       0.76      0.70      0.73       142

    accuracy                           0.92       882
   macro avg       0.85      0.83      0.84       882
weighted avg       0.91      0.92      0.91       882

Parameter Terbaik: {'model__min_samples_split': 10, 'model__min_samples_leaf': 2, 'model__max_depth': 10, 'model__criterion': 'gini'}


In [353]:
eval_classification(random_search_dt)

Accuracy (val Set): 0.92
Accuracy (Train Set): 0.97
Precision (val Set): 0.76
Recall (val Set): 0.70
F1-Score (val Set): 0.73
roc_auc (val): 0.91
roc_auc (train): 0.98
recall (cv train): 0.8097774697393498
recall (cv test): 0.7454052989264256


In [354]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_val, random_search_dt.predict(X_val)))

[[709  31]
 [ 43  99]]


In [332]:
# best_model = random_search_dt.best_estimator_.named_steps['model']
# show_feature_importance(best_model)

### RANDOM FOREST

In [358]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(
        random_state=42, 
        class_weight='balanced', 
        max_depth=10,            
        min_samples_leaf=5, 
        n_estimators=100))
])

pipeline_rf.fit(X_train, y_train)
y_pred = pipeline_rf.predict(X_val)

print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.99      0.97       740
           1       0.92      0.75      0.83       142

    accuracy                           0.95       882
   macro avg       0.94      0.87      0.90       882
weighted avg       0.95      0.95      0.95       882



In [359]:
eval_classification(pipeline_rf)

Accuracy (val Set): 0.95
Accuracy (Train Set): 0.99
Precision (val Set): 0.92
Recall (val Set): 0.75
F1-Score (val Set): 0.83
roc_auc (val): 0.97
roc_auc (train): 1.00
recall (cv train): 0.95955964256541
recall (cv test): 0.9114251945237861


In [334]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_val, pipeline_rf.predict(X_val)))

[[731   9]
 [ 35 107]]


#### Parameter Tuning

In [335]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# pipeline
pipeline_rf = Pipeline([
    ('model', RandomForestClassifier(
        random_state=42, 
        class_weight='balanced'))
])

param_grid_rf = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 20, None],
    'model__min_samples_leaf': [1, 2, 5]
}

# grid search
grid_search_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring='recall',
    n_jobs=-1
)

# training (pakai data asli, bukan hasil SMOTE!)
grid_search_rf.fit(X_train, y_train)

# prediksi
y_pred_rf = grid_search_rf.predict(X_val)

print(classification_report(y_val, y_pred_rf))
print("Parameter Terbaik:", grid_search_rf.best_params_)

              precision    recall  f1-score   support

           0       0.99      1.00      1.00       740
           1       1.00      0.96      0.98       142

    accuracy                           0.99       882
   macro avg       1.00      0.98      0.99       882
weighted avg       0.99      0.99      0.99       882

Parameter Terbaik: {'model__max_depth': 20, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}


In [360]:
eval_classification(grid_search_rf)

Accuracy (val Set): 0.99
Accuracy (Train Set): 1.00
Precision (val Set): 1.00
Recall (val Set): 0.96
F1-Score (val Set): 0.98
roc_auc (val): 0.99
roc_auc (train): 1.00
recall (cv train): 1.0
recall (cv test): 0.9957844971929479


In [361]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_val, grid_search_rf.predict(X_val)))

[[740   0]
 [  5 137]]


### XGBOOST

In [362]:
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# pipeline
pipeline_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('model', XGBClassifier(
        random_state=42,
        eval_metric='logloss'
    ))
])

pipeline_xgb.fit(X_train, y_train)
y_pred = pipeline_xgb.predict(X_val)

print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       740
           1       1.00      0.98      0.99       142

    accuracy                           1.00       882
   macro avg       1.00      0.99      0.99       882
weighted avg       1.00      1.00      1.00       882



In [363]:
eval_classification(pipeline_xgb)

Accuracy (val Set): 1.00
Accuracy (Train Set): 1.00
Precision (val Set): 1.00
Recall (val Set): 0.98
F1-Score (val Set): 0.99
roc_auc (val): 1.00
roc_auc (train): 1.00
recall (cv train): 1.0
recall (cv test): 0.995774647887324


In [338]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_val, pipeline_xgb.predict(X_val)))

[[740   0]
 [  3 139]]


#### Parameter Tuning

In [339]:
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report

# pipeline
pipeline_xg = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('model', XGBClassifier(
        random_state=42,
        eval_metric='logloss'
    ))
])

# hyperparameter (pakai prefix model__)
param_dist_xg = {
    'model__n_estimators': [100, 200, 300, 500],
    'model__max_depth': [3, 5, 7, 10],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__subsample': [0.6, 0.8, 1.0],
    'model__colsample_bytree': [0.6, 0.8, 1.0],
    'model__gamma': [0, 0.1, 0.2, 0.3],
    'model__min_child_weight': [1, 3, 5]
}

# randomized search
random_search_xg = RandomizedSearchCV(
    estimator=pipeline_xg,
    param_distributions=param_dist_xg,
    n_iter=25,
    cv=5,
    scoring='f1',  # bisa diganti recall kalau fokus attrition
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# training (pakai data asli!)
random_search_xg.fit(X_train, y_train)

# prediksi
y_pred_xg = random_search_xg.predict(X_val)

# evaluasi
print(classification_report(y_val, y_pred_xg))

Fitting 5 folds for each of 25 candidates, totalling 125 fits
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       740
           1       1.00      0.96      0.98       142

    accuracy                           0.99       882
   macro avg       1.00      0.98      0.99       882
weighted avg       0.99      0.99      0.99       882



In [364]:
eval_classification(random_search_xg)

Accuracy (val Set): 0.99
Accuracy (Train Set): 1.00
Precision (val Set): 1.00
Recall (val Set): 0.96
F1-Score (val Set): 0.98
roc_auc (val): 0.99
roc_auc (train): 1.00
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Fitting 5 folds for each of 25 candidates, totalling 125 fits
recall (cv train): 1.0
recall (cv test): 0.9915591450802719


In [340]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_val, random_search_xg.predict(X_val)))

[[740   0]
 [  5 137]]
